# DECIPHER Summer School 2025, Bilbao - DIVACoast.jl
by Global Climate Forum, Berlin 2025

---

## What is DIVACoast.jl? 

`DIVACoast.jl` is a package for the programming language Julia. The package is currently under development and designed to provide a broad toolset for **coastal risk & adapatation assessment**. The key concept of `DIVACoast.jl` is the concept of risk. Following the definition of the Intergovernmental Panel on Climate Change (IPCC), risk constituted by the three components of hazard, exposure and vulnerability (Oppenheimer et al., 2019; Wong et al., 2014). While on the long run the package is meant to serve multiple coastal risks including the risk of flooding, erosion, salinity intrusion and wetland change, the current release concentrates on flood risk.

## What will you learn in this session?
1. Coastal Risk assessment using DIVACoast.jl
2. Flood Exposure assessment
    - calculate exposure using `HypsometricProfiles`
    - simulate socio-economic developments
3. Flood Damage calculation
    - Depth Damage Functions
    - assess expected damages
4. Flood adaptation
    - Hard defences
    - Accommodation 
    - Nature Based Solutions

---


## Setup

### On Google Colab

In [ ]:
run(`sh -c "rm DIVACoast.jl -rf && rm DIVACoastColab -rf && git clone https://github.com/GlobalClimateForum/DIVACoastColab.git && git clone https://github.com/GlobalClimateForum/DIVACoast.jl.git"`)

then...

In [ ]:
cd("/content")
using Pkg; Pkg.activate("./DIVACoastColab/"); Pkg.instantiate() # Requirements
cd("./DIVACoast.jl"); include("./src/DIVACoast.jl"); using .DIVACoast # Load DIVACoast.jl

### Load Required Packages

In [ ]:
using Plots
using DataFrames

---

## Exposure assessment

### What is a Hypsometric Profile?
Currently the main way to represent exposure in DIVACoast.jl is as `HypsometricProfile`. This is a special kind of coastal profile that allows for a very efficient computation of flood damages, which is beneficial for running large number of damage assessments as, e.g., required for many economic questions that involve optimization.

### Hypsometric Profiles

For this session we pre-calculated `HypsometricProfile` around Europe. You can load them using:

In [ ]:
testdata    = "./testdata/hpfs_europe.jld2"
hypsometric_profiles = DIVACoast.load(testdata, Dict{Int32, HypsometricProfile{Float32}})

Each floodplain is associated with a `HypsometricProfile`. You can get a floodplain of interest (your home town or favorite holiday destination?) by copying the floodplain ID from this **[map](https://globalclimateforum.github.io/DIVACoastExplorer/)**. <br> **Note:** The flooplain outlines in the map are simplified in order to visualize them in a efficient manner, they do not reflect the real floodplain outlines accurate.

In [ ]:
floodplain = hypsometric_profiles[75804]

... and plot it. Check the plot and try to think about what it represents.

In [ ]:
plot(floodplain)

to inspect the exposure values (assets and population) you can convert the `HypsometricProfile` to a `DataFrame`:

In [ ]:
floodplain_df = DataFrame(floodplain)

using the DataFrame you can plot the different exposure variables from the Hypsometric Profile.

In [ ]:
plot(floodplain_df.cumulativeArea, floodplain_df.elevation, 
    xlabel = "cumulative Area (km²)", ylabel = "Elevation (m)",
    label = "exposed area",
    legend = :topright)

### Exposure

The `exposure()` function is used to calculate the cumulative exposure below given water level (`wl`) for a hypsometric profile and for all exposure variables in a `HypsometricProfile` given an `InundationModel`. 

Arguments are:
- `hspf::HypsometricProfile{DT}`: The hypsometric profile with elevation, area and exposure data.
- `wl::Real`: The water level (surge height) for which exposure is calculated.
- `InundationModel`: inundation model used to calculate the propagation of the water level. The default is bathtub model.

#### Excercise 1

Use the `exposure()` function to calculate exposed assets and population under different flood scenarios: 1m, 2m , 3m ... 10m for your floodplain of interest.

In [ ]:
# Type your code here

#### Solution Excercise 1

In [ ]:
waterlevel_scenarios = 1:1:10 |> Vector{Float32}

exposures = map(wl -> wl =>  exposure(floodplain, wl), waterlevel_scenarios)
# exposures = map(wl -> wl => named(exposure, (floodplain, wl)), waterlevel_scenarios) # To get named exposures
exposures

#### Excercise 2

Modify the exposure in a `HypsometricProfile` you can use the [HypsometricProfile modifiers](https://globalclimateforum.github.io/DIVACoast.jl/exposure.html#Modifying-Hypsometric-Profiles) provided by `DIVACoast.jl`. Assume the following scenario: Until 2035 population declines by a factor of facotor of **0.85 above 5m** and population increases by a factor of **1.1 below 5m**.

In [ ]:
# Type your code here

#### Solution Excercise 2

In [ ]:
multiply_exposure_above!(floodplain, 5.0, :population,  0.85) # Applies a factor of 0.85 to population above 5m elevation
multiply_exposure_below!(floodplain, 5.0, :population, 1.1) # Applies a factor of 1.1 to population below 5m elevation

#### Excercise 3 (additional)

Simulate the following scenarios: There is an increase in population and assets in lower elevation coastal areas below 5m until 2050. Higher elevations above 5 meters over sea level experience a decrease in population and assets. Use the modifiers to implement the following scenarios:

| year | pop. change <5m | pop. change ≥5m | asset change <5m | asset change ≥5m |
| ---- | --------------- | --------------- | ---------------- | ---------------- |
| 2025 | 1.00            | 1.00            | 1.00             | 1.00             |
| 2030 | 1.01            | 0.98            | 1.02             | 1.00             |
| 2035 | 1.03            | 0.96            | 1.04             | 0.99             |
| 2040 | 1.05            | 0.94            | 1.06             | 0.97             |
| 2045 | 1.06            | 0.92            | 1.08             | 0.95             |
| 2050 | 1.07            | 0.90            | 1.10             | 0.94 

In [ ]:
# To save you some typing
population_change_below_5m = [1.0, 1.01, 1.03, 1.05, 1.06, 1.07]
population_change_above_5m = [1.0, 0.98, 0.96, 0.94, 0.92, 0.90]
asset_change_below_5m = [1.0, 1.02, 1.04, 1.06, 1.08, 1.10]
asset_change_above_5m = [1.0, 1.0, 0.99, 0.97, 0.95, 0.94]
years = Vector{Int32}(2025:5:2050)

# You can also use rates, keep in mind that modifiers are mutating the HypsometricProfile structure
yrly_rate = (yrly_factors) -> round.(diff(yrly_factors), digits = 2) 

# Type your code here

---

## Damage assessment

In contrast to exposure, **damage** is dependend on **inundation depth**. Damage is a function of Flood Hazard, Flood Exposure, and Vulnerability. Hazard being extreme waterlevels, exposure the assets and population at risk, and  $vulnerability$ is represeneted by a Depth Damage Function. A Depth Damage function relates inundation depth to the share of the asset values that get damaged.

A common example is the following depth damage function

$$ 
vulnerability(\text{depth}) = \frac{\text{depth}}{\text{depth} + 1} 
$$

Implemented as:

In [ ]:
vulnerability = depth -> depth / (depth + 1)

depths = Vector{Float32}(1:1:10)
plot(depths, vulnerability.(depths), xlabel = "Inundation Depth (m)", ylabel = "Damage Share", label = "Damage Function",
    legend = :topleft)

To calulcate the damage within a floodplain for a given waterlevel you can use the `damage()` function:

```julia
damage(floodplain::HypsometricProfile, wl<:Real, exposure_variable::Symbol, ddf::Function)
```

#### Excercise 4

Calculate the damage to assets for different flood events (1.0m, 2.0m, ...) and compare it to the calculated exopsure (ex. 1). You can use the depth damage function $v$ from above or define your own. 


In [ ]:
# Type your code here

#### Solution Excercise 4

In [ ]:
#  Define waterlevel scenarions in 1m increments
waterlevel_scenarios =Vector{Float32}(1:1:10) 
# appply damage function to waterlevel scenarios
damages = map(wl -> wl => damage(floodplain, wl, :assets, vulnerability), waterlevel_scenarios) 

---

## Protection

Coastal protection is characterised by reducing risk to coastal assets and populations by decreasing the hazard of sea level rise and coastal flooding. Physical infrastructure such as dikes, flood barriers or dams hold back extreme water levels. For the next excercise, let's consider the case of **Amsterdam  (id: 65562)**. Amsterdam is characterized by a large floodplain which is home to around 1 million people and high GDP per capita. In addition, large storm surges as are common in the North Sea as shown by the following extreme waterlevel distribution:


| return period   | waterlevel (m) |
| --------------- | -------------- |
| 1 in 1 year     | 2.84           |
| 1 in 5 years    | 3.20           |
| 1 in 10 years   | 3.33           |
| 1 in 25 years   | 3.47           |
| 1 in 50 years   | 3.56           |
| 1 in 100 years  | 3.63           |
| 1 in 500 years  | 3.80           |
| 1 in 1000 years | 3.87           |

To estimate the best fitting Generalized Extreme Value distribution (Gumbel, Frechet, Weibull) you can use the `estimate_gev_distribution()` as in the following: 

In [ ]:
water_levels = [2.84, 3.20, 3.33, 3.47, 3.56, 3.63, 3.80, 3.87]
return_periods = [1, 5, 10, 25, 50, 100, 500, 1000]

# Covert return periods to cumulative probabilities
return_probs = map(rp -> 1 - (1 / rp), return_periods)

# Estimate GEV distribution parameters (assuming the function returns a distribution object)
gev_dist, _ = estimate_gev_distribution(water_levels, return_probs)

What do we do with this distribution? In order to assess optimal adaptation strategies today we need a measure of risk in the future. One such measure are **expected annual damages**.

With the `expected_damage` function you can compute the average annual damages for all possible flood events by integrating the damages with their probability of occurence. 

```julia
# Calculate expected annual damages to assets at a protection height of 2m
expected_damage(amsterdam_fp, gev_dist, 2.0, :assets, vulnerability)
```

#### Excercise 5 - Protection

The goal of this excercise is to implement a simple optimization problem in order to find the protection level at which expected annual damages for Amsterdam are halved (compared to a hypothetical scenario where Amsterdam is completely unprotected).

To solve this problem, in a first step calculate the expected annual damages to assets in Amsterdam (id : 65562) under the assumption that there is no protection in place. Second, assess at which protection level (flood return period) would expected annual damages be halved?

In [ ]:
# Type your code here

#### Solution Excercise 5

In [ ]:
amsterdam_fp = hypsometric_profiles[65562]

expected_damages_0 = expected_damage(amsterdam_fp, gev_dist, 0.0, :assets, vulnerability)

println("Expected damages at 0m protection level: ", expected_damages_0)

current_protection_level = 0.0
expected_damages = Inf

while expected_damages > expected_damages_0 / 2
    current_protection_level += 0.1
    expected_damages = expected_damage(amsterdam_fp, gev_dist, current_protection_level, :assets, vulnerability)
    println("Expected damages at protection level ", current_protection_level, "m: ", expected_damages)
end

println(current_protection_level)

### Accomodation

Contrary to coastal protection measures, accomodation is often performed at the micro level and includes a range of architectural or institutional responses such as flood proofing or elevating buildings, uptake of insurance or policies to reduce the harm that floods cause. Unlike coastal protection, accomodation measures do not modify  flood hazard, but rather reduce the vulnerability of coastal populations and assets to flood impacts.


#### Excercise 6 - Accomodation

The purpose of this excercise is to implement a simple model of accomodation within the previous modeling framework. Which component of the risk model (hazard, exposure, vulnerability) would you modify to implement flood proofing of buildings of up to 1m? You can use the floodplain you previously selected or keep the Amsterdam example.

In [ ]:
# Type your code here

#### Solution Excercise 6

In order to model a reduction of vulnerability you can modify the depth-damage function. For example you can define a new function that reduces the damage to 0 for inundation depths smaller or equal to 1m, such that

$$
vulnerability(depth) = 
\begin{cases}
0 & \text{if } depth < 1 \\
\frac{depth}{depth + 1} & \text{else}
\end{cases}
$$

In [ ]:
# Depth-damage function that reduces the damage to 0 for inundation depths smaller or equal to 1m
vulnerability_reduced = depth -> depth <= 1 ? 0 : depth / (depth + 1)

# Apply the modified vulnerability function to the floodplain
damages_reduced = map(wl -> wl => expected_damages(floodplain, wl, :assets, vulnerability_reduced), waterlevel_scenarios)

## Additional
So far, we have only used the default Inundation Model in DIVACoast.jl, the "Bathtub" Inundation. The Bathtub model is useful for modeling flooding across large regional scales or for testing a broad variety of adaptation strategies in a runtime-efficient manner. For certain adaptation strategies, such as the implementation of wetlands, you may want to consider attenuation. Therefore, DIVACoast offers the attenuated Bathtub model. You can pass the attenuated Bathtub model as a keyword argument to the `exposure()` and `damage()` functions. The `LinearDistanceAttenuatedInundation` model takes an attenuation value as a parameter (m/km attenuation).

```julia
damage(floodplain, wl, :assets, vulnerability, LinearDistanceAttenuatedInundation(0.1))
```